# 08. DeepSeek LangChain MCP Client

## 01. 安装依赖

In [4]:

%%capture #避免显示pip安装信息
%pip install langchain-mcp-adapters langchain-deepseek python-dotenv langgraph

## 2. 自定义MCP Server

> 参考：[langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)

### 2.1 安装依赖

In [7]:
%%capture #避免显示pip安装信息
%pip install fastmcp

### 2.2 核心代码

In [ ]:
# math_server.py 见:./math_server.py，这里暂不支持执行
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

## Client 客户端

In [2]:
# Create server parameters for stdio connection
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent

from langchain_deepseek import ChatDeepSeek

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

model = ChatDeepSeek(model="deepseek-chat")

server_params = StdioServerParameters(
    command="python",
    # Make sure to update to the full absolute path to your math_server.py file
    args=["./math_server.py"],
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        # Create and run the agent
        agent = create_react_agent(model, tools)
        agent_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})
        print("Answer:", agent_response)

Answer: {'messages': [HumanMessage(content="what's (3 + 5) x 12?", additional_kwargs={}, response_metadata={}, id='c2533d58-dc51-43c1-9bfa-515ef22217c1'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_0587925b-8405-455b-92ab-058b79be58fc', 'function': {'arguments': '{"a": 3, "b": 5}', 'name': 'add'}, 'type': 'function', 'index': 0}, {'id': 'call_1_ed14c7eb-fc77-4fc2-b1ad-f838f9b6caec', 'function': {'arguments': '{"a": 8, "b": 12}', 'name': 'multiply'}, 'type': 'function', 'index': 1}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 218, 'total_tokens': 266, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 26}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': 'df64c236-87df-4848-97a5-d6bb7d5476c9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-73592cd2-

## 3. 多个 MCP 服务器

In [ ]:
from typing import List
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"

if __name__ == "__main__":
    mcp.run(transport="sse")

## 4. 多个MCP 集成Client

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat")

async with MultiServerMCPClient(
    {
        "math": {
            "command": "python",
            # Make sure to update to the full absolute path to your math_server.py file
            "args": ["./math_server.py"],
            "transport": "stdio",
        },
        "weather": {
            # make sure you start your weather server on port 8000
            "url": "http://localhost:8000/sse",
            "transport": "sse",
        }
    }
) as client:
    agent = create_react_agent(model, client.get_tools())
    math_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})
    print(math_response)
    weather_response = await agent.ainvoke({"messages": "what is the weather in nyc?"})
    print(weather_response)

{'messages': [HumanMessage(content="what's (3 + 5) x 12?", additional_kwargs={}, response_metadata={}, id='ba98248d-aa79-45ac-a90b-fee5395c4981'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_7bf5d54e-9c59-44bd-9136-da7c49717200', 'function': {'arguments': '{"a": 3, "b": 5}', 'name': 'add'}, 'type': 'function', 'index': 0}, {'id': 'call_1_5c3c8c75-3981-4c9f-9f58-a816b47d7de6', 'function': {'arguments': '{"a": 8, "b": 12}', 'name': 'multiply'}, 'type': 'function', 'index': 1}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 296, 'total_tokens': 344, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 104}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': '6c0e51f1-cc9f-4282-8c26-2a69d1e9fb83', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-f6eaf98d-e8f5-40

## 04. 对接 mcp.so

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

from langchain_deepseek import ChatDeepSeek
import os
import json
from dotenv import load_dotenv

load_dotenv()

model = ChatDeepSeek(model="deepseek-chat")

async with MultiServerMCPClient(
    {
        "tavily-mcp": {
            "command": "npx",
            "args": [
                "-y",
                "tavily-mcp@0.1.4"
            ],
            "env": {
                "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")
            },
            "autoApprove": [],
            "name": "tavily-mcp"
        },
        "amap-maps": {
            "command": "npx",
            "args": [
                "-y",
                "@amap/amap-maps-mcp-server"
            ],
            "env": {
                "AMAP_MAPS_API_KEY": os.getenv("AMAP_MAPS_API_KEY")
            }
        }
    }
) as client:
    agent = create_react_agent(model, client.get_tools())
    web_search_response = await agent.ainvoke({"messages": "2025年4月有哪些AI的重大新闻？"})
    print(f"原始返回内容：{web_search_response}")
    print(web_search_response["messages"][-1].content)
    print("------------------------------------------------------")
    gaode_lbs_response = await agent.ainvoke({"messages": "请帮我规划一条从北京到上海的自驾游路线，途经南京和杭州，时间为3天，尽量避开拥堵的路段"})
    print(f"原始返回内容：{gaode_lbs_response}")
    print(gaode_lbs_response["messages"][-1].content)

### 返回信息

原始返回内容：
```json
{'messages': [HumanMessage(content='2025年4月有哪些AI的重大新闻？', additional_kwargs={}, response_metadata={}, id='9b130c9a-ca92-4f08-8fc3-ba48bbaae647'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_bfcbc7c1-08e8-4de2-8336-c6b09ae79e03', 'function': {'arguments': '{"query":"2025年4月 AI 重大新闻","search_depth":"advanced","topic":"news","days":30,"max_results":10}', 'name': 'tavily-search'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 2774, 'total_tokens': 2820, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2752}, 'prompt_cache_hit_tokens': 2752, 'prompt_cache_miss_tokens': 22}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': '9d58f5d8-9113-4389-8d0b-115650a5d58b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-08eaf84a-b05a-49cd-96b3-c8628f7add58-0', tool_calls=[{'name': 'tavily-search', 'args': {'query': '2025年4月 AI 重大新闻', 'search_depth': 'advanced', 'topic': 'news', 'days': 30, 'max_results': 10}, 'id': 'call_0_bfcbc7c1-08e8-4de2-8336-c6b09ae79e03', 'type': 'tool_call'}], usage_metadata={'input_tokens': 2774, 'output_tokens': 46, 'total_tokens': 2820, 'input_token_details': {'cache_read': 2752}, 'output_token_details': {}}), ToolMessage(content='Detailed Results:\n\nTitle: 科技资讯AI速递：昨夜今晨科技热点一览丨2025年4月11日 - 新浪新闻\nURL: https://news.sina.com.cn/zx/ds/2025-04-11/doc-inestvsi2877509.shtml\nContent: ã\x80\x80ã\x80\x802025å¹´GSMA MWCä¸\x8aæµ·å¤§ä¼\x9aå°\x86è\x81\x9aç\x84¦AIä¸\x8e5Gå\x95\x86ä¸\x9aå\x8c\x96ç\xad\x89å\x89\x8dæ²¿è®®é¢\x98ã\x80\x82GSMAé¦\x96å¸\xadæ\x89§è¡\x8cå®\x98æ´ªæ\x9b\x9cåº\x84æ\x8c\x87å\x87ºï¼\x8cä¸\xadå\x9b½ç§\x91æ\x8a\x80ä¼\x81ä¸\x9aå\x9c¨å·´å¡\x9eç½\x97é\x82£MWCä¸\x8aè¡¨ç\x8e°æ\x8a¢ç\x9c¼ï¼\x8cå\x8d\x8eä¸ºã\x80\x81ä¸\xadå\x85´ã\x80\x81å°\x8fç±³ç\xad\x89270å®¶ä¼\x81ä¸\x9aå±\x95ç¤ºäº\x86AIã\x80\x815Gå\x8f\x8aç\x89©è\x81\x94ç½\x91é¢\x86å\x9f\x9fç\x9a\x84å\x88\x9bæ\x96°çª\x81ç\xa0´ï¼\x8cå\x85¶ä¸»è\x88\x9eå\x8f°æ¼\x94è®²å\x92\x8cGTIå³°ä¼\x9aè¡¨ç\x8e°å¤\x87å\x8f\x97ç\x9e©ç\x9b®ï¼\x8cé¢\x84ç¤ºç\x9d\x80ä¸\x8aæµ·ç«\x99å°\x86å»¶ç»\xadè¿\x99ä¸\x80æ\x8a\x80æ\x9c¯å±\x95ç¤ºç\x83\xadæ½®ã\x80\x82\n\nã\x80\x80ã\x80\x803 å\x88\x86é\x92\x9fè\x87ªå\x8a¨æ£\x80æµ\x8b 1000 å¤\x9aé¡¹å\x8a\x9fè\x83½ï¼\x8cè\x94\x9aæ\x9d¥å·¥å\x8e\x82è\x87ªç\xa0\x94â\x80\x9cå¤©æ\x8e¢â\x80\x9dAI è\x87ªæ£\x80ç³»ç»\x9fäº®ç\x9b¸ [...] ã\x80\x80ã\x80\x80é\x95¿å®\x89æ±½è½¦ä¸\x8eè\x85¾è®¯ç\xad¾ç½²æ·±å\x8c\x96æ\x99ºè\x83½å\x8c\x96å\x90\x88ä½\x9cå\x8d\x8fè®®\n\nã\x80\x80ã\x80\x80é\x95¿å®\x89æ±½è½¦ä¸\x8eè\x85¾è®¯äº\x8e2025å¹´4æ\x9c\x8810æ\x97¥å\x9c¨é\x87\x8dåº\x86ç\xad¾ç½²æ·±å\x8c\x96æ\x99ºè\x83½å\x8c\x96å\x90\x88ä½\x9cå\x8d\x8fè®®ï¼\x8cå\x8f\x8cæ\x96¹å°\x86æ\x95´å\x90\x88å\x90\x84è\x87ªæ\x8a\x80æ\x9c¯ä¼\x98å\x8a¿ï¼\x8cé\x87\x8dç\x82¹æ\x8e¨è¿\x9bæ\x99ºè\x83½å\x8c\x96æ\x95°æ\x8d®ä½\x93ç³»å»ºè®¾ï¼\x8cå\x85±å\x90\x8cæ\x9e\x84å»ºå®\x89å\x85¨ç¨³å®\x9aç\x9a\x84æ\x95°æ\x8d®é\x97\xadç\x8e¯ç\x94\x9fæ\x80\x81ï¼\x8cå\x8a\xa0é\x80\x9fæ±½è½¦æ\x99ºè\x83½å\x8c\x96å\x8f\x91å±\x95è¿\x9bç¨\x8bã\x80\x82ï¼\x88135å\xad\x97ï¼\x89\n\nã\x80\x80ã\x80\x80GSMA MWC ä¸\x8aæµ·äº®ç\x82¹å±\x95æ\x9c\x9bï¼\x9aè\x81\x9aç\x84¦ AIã\x80\x815G å\x8f\x98ç\x8e°ç\xad\x89 [...] ã\x80\x80ã\x80\x802025å¹´ç\x88\x86ç\x81«ç\x9a\x84â\x80\x9cå°\x8fæ\x99ºAIâ\x80\x9då¼\x80æº\x90é¡¹ç\x9b®ï¼\x8cå\x87\xadå\x80\x9få\x85¶æ\x8b\x9fäººå\x8c\x96è¯\xadé\x9f³äº¤äº\x92å\x92\x8cä½\x8eé\x97¨æ§\x9bç\x89¹æ\x80§ï¼\x8cä¸¤ä¸ªæ\x9c\x88å\x86\x85æ\x8e¥å\x85¥è®¾å¤\x87æ¿\x80å¢\x9e10ä¸\x87å\x8f°ã\x80\x82è¿\x99æ¬¾å\x9fºäº\x8eä¹\x90é\x91«ESP32è\x8a¯ç\x89\x87ç\x9a\x84AIç¡¬ä»¶ï¼\x8cç\x94±æ\x95\x99è\x82²å\x85¬å\x8f¸å\x8d\x81æ\x96¹è\x9e\x8dæµ·å\x9b¢é\x98\x9få¼\x80å\x8f\x91ï¼\x8cä»¥å\x85\x8dè´¹è\x81\x8aå¤©å\x8a\x9fè\x83½ä¸ºæ\xa0¸å¿\x83å\x90¸å¼\x95å¤§é\x87\x8fé\x9d\x9eä¸\x93ä¸\x9aç\x94¨æ\x88·å\x8f\x82ä¸\x8eDIYã\x80\x82å\x85¶300æ¯«ç§\x92ç\x9a\x84å¿«é\x80\x9få\x93\x8dåº\x94å\x92\x8cæ\x83\x85æ\x84\x9få\x8c\x96äº¤äº\x92å¼\x95å\x8f\x91å\x85¨ç½\x91ç\x83\xadè®®ï¼\x8cå¸¦å\x8a¨è\x8a¯ç\x89\x87å\x8e\x82å\x95\x86ã\x80\x81æ\x96\x87å\x88\x9bå\x93\x81ç\x89\x8cç\xad\x89äº§ä¸\x9aé\x93¾å\x8a\xa0å\x85¥ã\x80\x82ç\x8e°è±¡çº§ä¼\xa0æ\x92\xadè\x83\x8cå\x90\x8eï¼\x8cå±\x95ç\x8e°äº\x86å¼\x80æº\x90æ¨¡å¼\x8få¯¹AIç¡¬ä»¶ç\x94\x9fæ\x80\x81ç\x9a\x84é\x87\x8då¡\x91æ½\x9cå\x8a\x9bï¼\x8cé¢\n\nTitle: 科技资讯AI速递：昨夜今晨科技热点一览丨2025年4月14日 - 新浪新闻\nURL: https://news.sina.com.cn/zx/ds/2025-04-14/doc-inetanuy2077574.shtml\nContent: ã\x80\x80ã\x80\x80ç¾\x8eç³»è\x8a¯ç\x89\x87é\x81\xadå\x85³ç¨\x8eå\x8f\x8då\x88¶ï¼\x8cå\x93ªäº\x9bå\x9b½äº§è\x8a¯ç\x89\x87ç\x8e\x87å\x85\x88å\x8f\x97ç\x9b\x8aï¼\x9f [...] æ\x96°é\x97»ä¸\xadå¿\x83\n\nç§\x91æ\x8a\x80èµ\x84è®¯AIé\x80\x9fé\x80\x92ï¼\x9aæ\x98¨å¤\x9cä»\x8aæ\x99¨ç§\x91æ\x8a\x80ç\x83\xadç\x82¹ä¸\x80è§\x88 ä¸¨2025å¹´4æ\x9c\x8814æ\x97¥\n\nã\x80\x80ã\x80\x80ç§\x91æ\x8a\x80æ\x97¥æ\x96°æ\x9c\x88å¼\x82ï¼\x8cå\x85¨ç\x90\x83å\x88\x9bæ\x96°ä¸\x8dæ\x96\xadå\x88·æ\x96°è¾¹ç\x95\x8cã\x80\x82æ\x88\x91ä»¬ä¸ºæ\x82¨æ±\x87æ\x80»æ\x98¨å¤\x9cä»\x8aæ\x99¨ç\x9a\x84ç§\x91æ\x8a\x80é¢\x86å\x9f\x9fæ\x9c\x80æ\x96°å\x8a¨å\x90\x91ï¼\x8cå¸¦æ\x82¨å¿«é\x80\x9fäº\x86è§£å\x89\x8dæ²¿æ\x8a\x80æ\x9c¯ã\x80\x81çª\x81ç\xa0´æ\x80§ç\xa0\x94ç©¶å\x8f\x8aè¡\x8cä¸\x9aè¶\x8bå\x8a¿ã\x80\x82\n\nã\x80\x80ã\x80\x80ç§\x91æ\x8a\x80å·¨å¤´æ\x83\x8aé\xad\x82å\x8d\x81æ\x97¥ï¼\x9aiPhoneæ\x97\xa0é\x9c\x80å¤§æ¶¨ä»·ï¼\x8cè\x8b¹æ\x9e\x9cä¸\x80å¤\x9cä»\x8eICUå\x88°KTV [...] ã\x80\x80ã\x80\x80æ\x98\x86ä»\x91ä¸\x87ç»´æ\x8e¨å\x87º Skywork-OR1 ç³»å\x88\x97æ¨¡å\x9e\x8bï¼\x9aå\x85¨é\x9d¢å¼\x80æ\x94¾ã\x80\x81å\x85\x8dè´¹ä½¿ç\x94¨ã\x80\x81å®\x8cå\x85¨å¼\x80æº\x90\n\nã\x80\x80ã\x80\x80æ\x98\x86ä»\x91ä¸\x87ç»´å®£å¸\x83æ\x8e¨å\x87ºå\x85¨æ\x96°Skywork-OR1ç³»å\x88\x97å¤§æ¨¡å\x9e\x8bï¼\x8cè¯¥ç³»å\x88\x97å\x9fºäº\x8eæ\xad¤å\x89\x8dSkywork-o1æ¨¡å\x9e\x8bè¿\xadä»£ä¼\x98å\x8c\x96ï¼\x8cä¸»æ\x89\x93ä¸\xadæ\x96\x87é\x80»è¾\x91æ\x8e¨ç\x90\x86è\x83½å\x8a\x9bã\x80\x82æ\xad¤æ¬¡å\x8d\x87çº§ç\x89\x88æ\x9c¬æ\x9c\x80å¤§ç\x89¹ç\x82¹æ\x98¯å\x85¨é\x9d¢å¼\x80æ\x94¾ã\x80\x81å®\x8cå\x85¨å¼\x80æº\x90ä¸\x94å\x85\x8dè´¹ä½¿ç\x94¨ï¼\x8cä½\x93ç\x8e°äº\x86å\x85¬å\x8f¸æ\x8e¨å\x8a¨AIæ\x8a\x80æ\x9c¯æ\x99®æ\x83\xa0å\x8c\x96ç\x9a\x84å\x86³å¿\x83ã\x80\x82è¯¥ç³»å\x88\x97æ¨¡å\x9e\x8bç\x94±å¤©å·¥å\x9b¢é\x98\x9fç\xa0\x94å\x8f\x91ï¼\x8cæ\x88\x96å°\x86è¿\x9bä¸\x80æ\xad¥é\x99\x8dä½\x8eä¼\x81ä¸\x9aä½¿ç\x94¨AIæ\x8a\x80æ\x9c¯ç\x9a\x84é\x97¨æ§\x9bã\x80\x82\n\nTitle: 2025 年4 月14 日高盛AI 预测报告深度解析：结合财报与市场动态看 ...\nURL: https://www.moomoo.com/hans/community/feed/114336466862089\nContent: 2025 年 4 月 14 日 高盛 AI 预测报告深度解析：结合财报与市场动态看门道\n\n热门市场机会\n\n特朗普重返白宮\n\n追踪降息周期下的投资机会\n\n财报季进行中\n\n- 没有更多了 -\n\n投资产品和服务由Moomoo Financial Inc.提供，Moomoo Financial Inc.是美国证券投资者保护公司(SIPC)及美国金融业管理局(FINRA)的成员。如需了解Moomoo Financial Inc.的相关信息，请访问美国金融业管理局（FINRA）的BrokerCheck网站。\n\n期权交易会带来巨大的风险，并不适合所有投资者。投资者在参与任何期权交易策略前必须阅读《标准化期权的特点和风险》。期权交易非常复杂，可能在较短的时间内损失全部投资。某些复杂的期权策略有额外风险，损失可能超过原始投资额。任何索赔证明文件，如适用，将按要求提供。\n\n免费交易仅适用于通过Moomoo Financial Inc.在美国市场进行交易的美国居民。此交易方式可能产生其他费用。更多信息，请访问moomoo.com/us/pricing [...] 所有投资均存在风险，包括潜在的资本金损失。电子化交易亦存在风险。交易系统响应状况可能会因为市场情况、系统表现或其他因素而受到影响。市场波动等因素可能会影响账户访问和交易执行。\n\n投资产品不受美国联邦存款保险公司（FDIC）保护，亦非银行存款，且可能贬值。\n\n本网站上提供的服务和产品受到适用法律和法规以及我们服务条款和政策的约束。本服务和产品不适用于所有客户和所有地区。本服务和产品不适用于法律不允许提供此类服务和产品地区。\n\nCopyright © 2025 Moomoo Technologies Inc. 版权所有\n\nTitle: 联合国报告：人工智能将发展为4.8万亿美元市场 - UN News\nURL: https://news.un.org/zh/story/2025/04/1137251\nContent: Language\n\n语言\n\n菜单\n\n语言\n\n搜索\n\nAudio and Subscription\n\n联合国报告：人工智能将发展为4.8万亿美元市场\n\nWeibo\nTwitter\n打印\n电子邮件\n\n联合国报告：人工智能将发展为4.8万亿美元市场\n\n联合国的一份报告预测，到2033年，人工智能将发展成为价值4.8万亿美元的全球市场，规模相当于德国当前的经济总量。然而，如果不采取紧急行动，人工智能可能加剧全球不平等，进一步扩大数字鸿沟。\n\n联合国贸易和发展组织于周四发布的《2025年技术与创新报告》指出，人工智能领域的资源和优势正日益向少数国家和企业集中，全球人工智能治理体系亟需更加包容和公平的框架。\n\n权力集中，治理失衡\n\n报告指出，全球40%的私人研发投资由仅100家公司主导，其中绝大多数总部设在美国和中国。这表明人工智能技术的发展正在高度集中于少数企业和经济体。\n\n与此同时，全球范围内有118个国家几乎完全被排除在人工智能治理讨论之外，其中大多数来自全球南方。这进一步加剧了技术发展的不平衡，使得许多国家在人工智能时代面临被边缘化的风险。\n\n机遇与挑战并存 [...] 报告建议采取全球共享计算资源的方案，建立一个让所有国家都能公平获取计算能力和人工智能工具的平台，从而减少技术资源的不平衡分配。\n\n此外，报告还提议建立人工智能领域的透明度和问责制框架，类似于目前广泛应用的环境、社会与治理标准，以确保人工智能可持续和负责任地发展。\n\n相关新闻\n\n巴黎峰会：联合国秘书长呼吁共建包容、公正的人工智能未来\n\n【前方视野】引领人工智能发展走向，联合国推动全球治理\n\n联合国呼吁各国携手治理人工智能\n\n往日新闻\n\n巴黎峰会：联合国秘书长呼吁共建包容、公正的人工智能未来\n\n联合国秘书长古特雷斯在周二在巴黎召开的人工智能行动峰会上直言不讳地指出：人类在迎接未来的路上“还远未准备好”。他强调，人工智能正以科幻般的速度由虚构走向现实，正在深刻改变我们的生活、工作与互动方式，同时在教育、医疗保健和农业等关键领域催生前所未有的突破，与此同时其潜在风险与责任也日益凸显。\n\n【前方视野】AI双刃剑：全球治理学者薛澜解读科技进步背后的隐忧与希望\n\nTitle: 朱皓：打破藩篱，加速AI走向物理世界 - 时政频道- 光明网\nURL: https://politics.gmw.cn/2025-04/14/content_37965679.htm\nContent: ç\x82¹å\x87»å\x8f³ä¸\x8aè§\x92å¾®ä¿¡å¥½å\x8f\x8b\n\næ\x9c\x8bå\x8f\x8bå\x9c\x88\n\n\n\nè¯·ä½¿ç\x94¨æµ\x8fè§\x88å\x99¨å\x88\x86äº«å\x8a\x9fè\x83½è¿\x9bè¡\x8cå\x88\x86äº«\n\n\n\nå\x85¨é\x83¨å¯¼è\x88ª\n\næ\x9c±ç\x9a\x93ï¼\x9aæ\x89\x93ç\xa0´è\x97©ç¯±ï¼\x8cå\x8a\xa0é\x80\x9fAIèµ°å\x90\x91ç\x89©ç\x90\x86ä¸\x96ç\x95\x8c\n\nã\x80\x80ã\x80\x80å\x85\x89æ\x98\x8eç½\x91è®¯ï¼\x88è®°è\x80\x85 æ\x9d\x8eé£\x9eï¼\x894æ\x9c\x8814æ\x97¥ï¼\x8c2025å¹´ä¸\x96ç\x95\x8cäº\x92è\x81\x94ç½\x91å¤§ä¼\x9aäº\x9aå¤ªå³°ä¼\x9aä¸»è®ºå\x9d\x9bæ\x9a¨æ\x95°æ\x99ºæ\x9cªæ\x9d¥è®ºå\x9d\x9bå\x9c¨ä¸\xadå\x9b½é¦\x99æ¸¯ä¸¾è¡\x8cã\x80\x82ç¾¤æ\xa0¸ç§\x91æ\x8a\x80è\x81\x94å\x90\x88å\x88\x9bå§\x8bäººå\x85¼é¦\x96å¸\xadæ\x8a\x80æ\x9c¯å®\x98æ\x9c±ç\x9a\x93è¡¨ç¤ºï¼\x8cäººå·¥æ\x99ºè\x83½æ\xad£å\x9c¨èµ\x8bè\x83½å\x8d\x83è¡\x8cç\x99¾ä¸\x9aï¼\x8cä½\x86å¹¶ä¸\x8dè\x83½å¾\x88å¥½èµ°è¿\x9bç\x89©ç\x90\x86ä¸\x96ç\x95\x8cã\x80\x82é\x9c\x80è¦\x81å\x80\x9få\x8a©å¼\x80æº\x90ç\x9a\x84å\x8a\x9bé\x87\x8fï¼\x8cå\x8a\xa0é\x80\x9fAIèµ°å\x90\x91ç\x89©ç\x90\x86ä¸\x96ç\x95\x8cã\x80\x82 [...] è§\x86è§\x89ç\x84¦ç\x82¹\n\nä¹\xa0è¿\x91å¹³ä¹\x98å\x9d\x90ç\x9a\x84ä¸\x93æ\x9cºé\x99\x8dè\x90½è¶\x8aå\x8d\x97æ²³å\x86\x85å\x86\x85æ\x8e\x92å\x9b½é\x99\x85æ\x9cºå\x9cº\n\nç¬¬äº\x94å±\x8aä¸\xadå\x9b½å\x9b½é\x99\x85æ¶\x88è´¹å\x93\x81å\x8d\x9aè§\x88ä¼\x9aä¸¾å\x8a\x9e\n\næ\x9c\x80ç\x83\xadæ\x96\x87ç«\n\nç\x8b¬å®¶ç\xad\x96å\x88\x92\n\n2025å¹´å\x85¨å\x9b½ä¸¤ä¼\x9a\n\n2025å¹´æ\x96°æ\x98¥èµ°å\x9fºå±\x82\n\nç¬¬ä¹\x9då±\x8aäº\x9aæ´²å\x86¬å\xad£è¿\x90å\x8a¨ä¼\x9a\n\nç\x83\xadç\x83\x88åº\x86ç¥\x9dæ¾³é\x97¨å\x9b\x9eå½\x92ç¥\x96å\x9b½25å\x91¨å¹´\n\n2025å¹´æ\x98¥è¿\x90\n\n12Â·4ç¬¬å\x8d\x81ä¸\x80ä¸ªå\x9b½å®¶å®ªæ³\x95æ\x97¥\n\nTitle: 金宪：AI不仅是重要技术，更是重塑全球经济的力量 - 时政频道\nURL: https://politics.gmw.cn/2025-04/14/content_37965771.htm\nContent: ç\x82¹å\x87»å\x8f³ä¸\x8aè§\x92å¾®ä¿¡å¥½å\x8f\x8b\n\næ\x9c\x8bå\x8f\x8bå\x9c\x88\n\n\n\nè¯·ä½¿ç\x94¨æµ\x8fè§\x88å\x99¨å\x88\x86äº«å\x8a\x9fè\x83½è¿\x9bè¡\x8cå\x88\x86äº«\n\n\n\nå\x85¨é\x83¨å¯¼è\x88ª\n\né\x87\x91å®ªï¼\x9aAIä¸\x8dä»\x85æ\x98¯é\x87\x8dè¦\x81æ\x8a\x80æ\x9c¯ï¼\x8cæ\x9b´æ\x98¯é\x87\x8då¡\x91å\x85¨ç\x90\x83ç»\x8fæµ\x8eç\x9a\x84å\x8a\x9bé\x87\x8f\n\nã\x80\x80ã\x80\x80å\x85\x89æ\x98\x8eç½\x91è®¯ï¼\x88è®°è\x80\x85 å\xad\x99æ»¡æ¡\x83ï¼\x894æ\x9c\x8814æ\x97¥ï¼\x8c2025å¹´ä¸\x96ç\x95\x8cäº\x92è\x81\x94ç½\x91å¤§ä¼\x9aäº\x9aå¤ªå³°ä¼\x9aä¸»è®ºå\x9d\x9bæ\x9a¨æ\x95°æ\x99ºæ\x9cªæ\x9d¥è®ºå\x9d\x9bå\x9c¨ä¸\xadå\x9b½é¦\x99æ¸¯ä¸¾è¡\x8cã\x80\x82è®ºå\x9d\x9bä¸\x8aï¼\x8cäº\x9aæ´²å\x9fºç¡\x80è®¾æ\x96½æ\x8a\x95èµ\x84é\x93¶è¡\x8cä»£ç\x90\x86å\x89¯æ\x80»è£\x81å\x85¼æ\x80»å¹²äº\x8bé\x87\x91å®ªè¡¨ç¤ºï¼\x8cAIä¸\x8dä»\x85æ\x98¯ä¸\x80é¡¹é\x87\x8dè¦\x81æ\x8a\x80æ\x9c¯ï¼\x8cæ\x9b´æ\x98¯é\x87\x8då¡\x91å\x85¨ç\x90\x83ç»\x8fæµ\x8eç\x9a\x84å\x8a\x9bé\x87\x8fã\x80\x82 [...] ã\x80\x80ã\x80\x80â\x80\x9cè¿\x99ç§\x8då\x8c\x85å®¹æ\x80§å\x8f\x91å±\x95ä¹\x9fæ\x98¯å\x85¨ç\x90\x83é\x83½åº\x94è¯¥è¦\x81é\x87\x8dè§\x86ç\x9a\x84ï¼\x8cå\x9c¨AIè¿\x99ä¸ªé¢\x86å\x9f\x9fï¼\x8cæ\x88\x91ä»¬é\x9c\x80è¦\x81æ\x9b´å¤\x9aç\x9a\x84å\x85¨ç\x90\x83å\x90\x88ä½\x9cï¼\x8cå\x90\x8cå\x90\x84å\x9b½é\x80\x9aå\x8a\x9bå\x90\x88ä½\x9cï¼\x8cå\x85±å\x90\x8cå\x8aªå\x8a\x9bæ\x9d¥å¡\x91é\x80\xa0æ\x9b´å¥½ç\x9a\x84AIæ\x9cªæ\x9d¥ã\x80\x82â\x80\x9dé\x87\x91å®ªè¡¨ç¤ºã\x80\x82\n\nç\x9b¸å\x85³é\x98\x85è¯»\n\n\n\n\n\næ\x82¨æ\xad¤æ\x97¶ç\x9a\x84å¿\x83æ\x83\n\nè§\x86è§\x89ç\x84¦ç\x82¹\n\nä¹\xa0è¿\x91å¹³ä¹\x98å\x9d\x90ç\x9a\x84ä¸\x93æ\x9cºé\x99\x8dè\x90½è¶\x8aå\x8d\x97æ²³å\x86\x85å\x86\x85æ\x8e\x92å\x9b½é\x99\x85æ\x9cºå\x9cº\n\nç¬¬äº\x94å±\x8aä¸\xadå\x9b½å\x9b½é\x99\x85æ¶\x88è´¹å\x93\x81å\x8d\x9aè§\x88ä¼\x9aä¸¾å\x8a\x9e\n\næ\x9c\x80ç\x83\xadæ\x96\x87ç«\n\nç\x8b¬å®¶ç\xad\x96å\x88\x92\n\n2025å¹´å\x85¨å\x9b½ä¸¤ä¼\x9a\n\n2025å¹´æ\x96°æ\x98¥èµ°å\x9fºå±\x82\n\nç¬¬ä¹\x9då±\x8aäº\x9aæ´²å\x86¬å\xad£è¿\x90å\x8a¨ä¼\x9a [...] ç\x83\xadç\x83\x88åº\x86ç¥\x9dæ¾³é\x97¨å\x9b\x9eå½\x92ç¥\x96å\x9b½25å\x91¨å¹´\n\n2025å¹´æ\x98¥è¿\x90\n\n12Â·4ç¬¬å\x8d\x81ä¸\x80ä¸ªå\x9b½å®¶å®ªæ³\x95æ\x97¥', name='tavily-search', id='72f9e817-ad4d-4ccb-b14c-4eea45f765bb', tool_call_id='call_0_bfcbc7c1-08e8-4de2-8336-c6b09ae79e03'), AIMessage(content='2025年4月，AI领域发生了多起重大新闻事件，以下是部分关键内容：\n\n1. **GSMA MWC上海大会聚焦AI与5G商业化**  \n   - 2025年GSMA MWC上海大会展示了AI与5G技术的创新突破，华为、中兴、小米等270家企业展示了AI、5G及物联网领域的最新成果。  \n   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-11/doc-inestvsi2877509.shtml)\n\n2. **长安汽车与腾讯深化智能合作**  \n   - 长安汽车与腾讯于4月10日签署深化合作协议，共同推进智能化数据体系建设，加速汽车智能化发展。  \n   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-11/doc-inestvsi2877509.shtml)\n\n3. **开源AI项目“小智AI”引发热潮**  \n   - 基于乐鑫ESP32芯片的开源AI硬件项目“小智AI”因其低门槛和情感化交互特性，两个月内接入设备激增10万台，带动芯片和文创产业链发展。  \n   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-11/doc-inestvsi2877509.shtml)\n\n4. **联合国报告：AI市场将达4.8万亿美元**  \n   - 联合国预测到2033年AI市场规模将达4.8万亿美元，但需警惕技术资源集中和全球不平等问题。  \n   - [详情链接](https://news.un.org/zh/story/2025/04/1137251)\n\n5. **高盛发布AI预测报告**  \n   - 高盛结合财报与市场动态，分析了AI在2025年的发展趋势和投资机会。  \n   - [详情链接](https://www.moomoo.com/hans/community/feed/114336466862089)\n\n6. **开源大模型Skywork-OR1发布**  \n   - 明万维推出完全开源且免费使用的Skywork-OR1系列大模型，旨在降低企业使用AI技术的门槛。  \n   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-14/doc-inetanuy2077574.shtml)\n\n这些新闻展示了AI技术在商业化、开源生态、国际合作等方面的快速发展。如果需要更详细的内容，可以点击相关链接查看。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 519, 'prompt_tokens': 9243, 'total_tokens': 9762, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 9216}, 'prompt_cache_hit_tokens': 9216, 'prompt_cache_miss_tokens': 27}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': '2f45fa8c-9c62-484b-9477-08a942cf8d99', 'finish_reason': 'stop', 'logprobs': None}, id='run-19cf374e-2712-405b-a062-42e8a66dab88-0', usage_metadata={'input_tokens': 9243, 'output_tokens': 519, 'total_tokens': 9762, 'input_token_details': {'cache_read': 9216}, 'output_token_details': {}})]}

2025年4月，AI领域发生了多起重大新闻事件，以下是部分关键内容：

1. **GSMA MWC上海大会聚焦AI与5G商业化**  
   - 2025年GSMA MWC上海大会展示了AI与5G技术的创新突破，华为、中兴、小米等270家企业展示了AI、5G及物联网领域的最新成果。  
   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-11/doc-inestvsi2877509.shtml)

2. **长安汽车与腾讯深化智能合作**  
   - 长安汽车与腾讯于4月10日签署深化合作协议，共同推进智能化数据体系建设，加速汽车智能化发展。  
   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-11/doc-inestvsi2877509.shtml)

3. **开源AI项目“小智AI”引发热潮**  
   - 基于乐鑫ESP32芯片的开源AI硬件项目“小智AI”因其低门槛和情感化交互特性，两个月内接入设备激增10万台，带动芯片和文创产业链发展。  
   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-11/doc-inestvsi2877509.shtml)

4. **联合国报告：AI市场将达4.8万亿美元**  
   - 联合国预测到2033年AI市场规模将达4.8万亿美元，但需警惕技术资源集中和全球不平等问题。  
   - [详情链接](https://news.un.org/zh/story/2025/04/1137251)

5. **高盛发布AI预测报告**  
   - 高盛结合财报与市场动态，分析了AI在2025年的发展趋势和投资机会。  
   - [详情链接](https://www.moomoo.com/hans/community/feed/114336466862089)

6. **开源大模型Skywork-OR1发布**  
   - 明万维推出完全开源且免费使用的Skywork-OR1系列大模型，旨在降低企业使用AI技术的门槛。  
   - [详情链接](https://news.sina.com.cn/zx/ds/2025-04-14/doc-inetanuy2077574.shtml)

这些新闻展示了AI技术在商业化、开源生态、国际合作等方面的快速发展。如果需要更详细的内容，可以点击相关链接查看。
------------------------------------------------------
原始返回内容：

```json
{'messages': [HumanMessage(content='请帮我规划一条从北京到上海的自驾游路线，途经南京和杭州，时间为3天，尽量避开拥堵的路段', additional_kwargs={}, response_metadata={}, id='d31ddf75-94ec-40a2-ad37-8477361f5076'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_34bfa41d-f7e3-4001-bac5-9c57406eb379', 'function': {'arguments': '{"address": "北京"}', 'name': 'maps_geo'}, 'type': 'function', 'index': 0}, {'id': 'call_1_ad48714c-16b3-40ff-a954-aa57e48bd2be', 'function': {'arguments': '{"address": "南京"}', 'name': 'maps_geo'}, 'type': 'function', 'index': 1}, {'id': 'call_2_12e1cb31-76ad-49f8-91a8-a859883a5d7a', 'function': {'arguments': '{"address": "杭州"}', 'name': 'maps_geo'}, 'type': 'function', 'index': 2}, {'id': 'call_3_3f4b2bca-a29d-4528-8abc-0feb13de4553', 'function': {'arguments': '{"address": "上海"}', 'name': 'maps_geo'}, 'type': 'function', 'index': 3}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 2791, 'total_tokens': 2868, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2752}, 'prompt_cache_hit_tokens': 2752, 'prompt_cache_miss_tokens': 39}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': '36965f27-a836-42d6-a036-a95d8d874243', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-726eb51d-4da7-47a3-804b-432283b483c1-0', tool_calls=[{'name': 'maps_geo', 'args': {'address': '北京'}, 'id': 'call_0_34bfa41d-f7e3-4001-bac5-9c57406eb379', 'type': 'tool_call'}, {'name': 'maps_geo', 'args': {'address': '南京'}, 'id': 'call_1_ad48714c-16b3-40ff-a954-aa57e48bd2be', 'type': 'tool_call'}, {'name': 'maps_geo', 'args': {'address': '杭州'}, 'id': 'call_2_12e1cb31-76ad-49f8-91a8-a859883a5d7a', 'type': 'tool_call'}, {'name': 'maps_geo', 'args': {'address': '上海'}, 'id': 'call_3_3f4b2bca-a29d-4528-8abc-0feb13de4553', 'type': 'tool_call'}], usage_metadata={'input_tokens': 2791, 'output_tokens': 77, 'total_tokens': 2868, 'input_token_details': {'cache_read': 2752}, 'output_token_details': {}}), ToolMessage(content='{\n  "return": [\n    {\n      "country": "中国",\n      "province": "北京市",\n      "city": "北京市",\n      "citycode": "010",\n      "district": [],\n      "street": [],\n      "number": [],\n      "adcode": "110000",\n      "location": "116.407387,39.904179",\n      "level": "省"\n    }\n  ]\n}', name='maps_geo', id='356dbedc-959d-4a41-a3c9-43e628fd2357', tool_call_id='call_0_34bfa41d-f7e3-4001-bac5-9c57406eb379'), ToolMessage(content='{\n  "return": [\n    {\n      "country": "中国",\n      "province": "江苏省",\n      "city": "南京市",\n      "citycode": "025",\n      "district": [],\n      "street": [],\n      "number": [],\n      "adcode": "320100",\n      "location": "118.796624,32.059344",\n      "level": "市"\n    }\n  ]\n}', name='maps_geo', id='858f4dd3-d5d5-418d-a85a-3dac338e1db3', tool_call_id='call_1_ad48714c-16b3-40ff-a954-aa57e48bd2be'), ToolMessage(content='{\n  "return": [\n    {\n      "country": "中国",\n      "province": "浙江省",\n      "city": "杭州市",\n      "citycode": "0571",\n      "district": [],\n      "street": [],\n      "number": [],\n      "adcode": "330100",\n      "location": "120.210792,30.246026",\n      "level": "市"\n    }\n  ]\n}', name='maps_geo', id='edfdb8de-cf50-4602-a2bb-4c7effc330a2', tool_call_id='call_2_12e1cb31-76ad-49f8-91a8-a859883a5d7a'), ToolMessage(content='{\n  "return": [\n    {\n      "country": "中国",\n      "province": "上海市",\n      "city": "上海市",\n      "citycode": "021",\n      "district": [],\n      "street": [],\n      "number": [],\n      "adcode": "310000",\n      "location": "121.473667,31.230525",\n      "level": "省"\n    }\n  ]\n}', name='maps_geo', id='99abd73e-3d71-4bf0-9614-9dc7ccedf4c3', tool_call_id='call_3_3f4b2bca-a29d-4528-8abc-0feb13de4553'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_aa42068c-0d52-4e84-90a7-d0078d4d495a', 'function': {'arguments': '{"origin": "116.407387,39.904179", "destination": "118.796624,32.059344"}', 'name': 'maps_direction_driving'}, 'type': 'function', 'index': 0}, {'id': 'call_1_b79b2e17-2f57-4b3c-89db-629ba972e825', 'function': {'arguments': '{"origin": "118.796624,32.059344", "destination": "120.210792,30.246026"}', 'name': 'maps_direction_driving'}, 'type': 'function', 'index': 1}, {'id': 'call_2_5c9a56d4-343f-4727-95bc-039915a3ad83', 'function': {'arguments': '{"origin": "120.210792,30.246026", "destination": "121.473667,31.230525"}', 'name': 'maps_direction_driving'}, 'type': 'function', 'index': 2}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 133, 'prompt_tokens': 3265, 'total_tokens': 3398, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 3264}, 'prompt_cache_hit_tokens': 3264, 'prompt_cache_miss_tokens': 1}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': '2f7c2498-80e1-46fe-bf64-40ff238cf22b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-bdeb94c0-a418-4e0d-a6e9-a3bec33317e4-0', tool_calls=[{'name': 'maps_direction_driving', 'args': {'origin': '116.407387,39.904179', 'destination': '118.796624,32.059344'}, 'id': 'call_0_aa42068c-0d52-4e84-90a7-d0078d4d495a', 'type': 'tool_call'}, {'name': 'maps_direction_driving', 'args': {'origin': '118.796624,32.059344', 'destination': '120.210792,30.246026'}, 'id': 'call_1_b79b2e17-2f57-4b3c-89db-629ba972e825', 'type': 'tool_call'}, {'name': 'maps_direction_driving', 'args': {'origin': '120.210792,30.246026', 'destination': '121.473667,31.230525'}, 'id': 'call_2_5c9a56d4-343f-4727-95bc-039915a3ad83', 'type': 'tool_call'}], usage_metadata={'input_tokens': 3265, 'output_tokens': 133, 'total_tokens': 3398, 'input_token_details': {'cache_read': 3264}, 'output_token_details': {}}), ToolMessage(content='{\n  "route": {\n    "origin": "116.407387,39.904179",\n    "destination": "118.796624,32.059344",\n    "paths": [\n      {\n        "distance": "1005711",\n        "duration": "39352",\n        "steps": [\n          {\n            "instruction": "沿正义路向北行驶407米右转",\n            "road": "正义路",\n            "distance": "407",\n            "orientation": "北",\n            "duration": "81"\n          },\n          {\n            "instruction": "沿东长安街向东行驶937米右转",\n            "road": "东长安街",\n            "distance": "937",\n            "orientation": "东",\n            "duration": "139"\n          },\n          {\n            "instruction": "沿崇文门内大街途径崇文门外大街、天坛路、天坛东路、蒲黄榆路、榴乡路向南行驶7.3千米靠左进入主路",\n            "road": "崇文门内大街",\n            "distance": "7310",\n            "orientation": "南",\n            "duration": "1145"\n          },\n          {\n            "instruction": "沿榴乡路途径德贤路、庑殿桥、旺兴湖桥、旧宫新桥、G3京台高速向南行驶12.0千米靠左沿主路行驶",\n            "road": "榴乡路",\n            "distance": "11979",\n            "orientation": "南",\n            "duration": "555"\n          },\n          {\n            "instruction": "沿G3京台高速向东南行驶144米靠左沿主路行驶",\n            "road": "G3京台高速",\n            "distance": "144",\n            "orientation": "东南",\n            "duration": "5"\n          },\n          {\n            "instruction": "沿G3京台高速向南行驶231米减速行驶到达收费站",\n            "road": "G3京台高速",\n            "distance": "231",\n            "orientation": "南",\n            "duration": "19"\n          },\n          {\n            "instruction": "沿G3京台高速途径大张本庄桥、伊庄北桥、京台大龙河桥、京台小龙河桥、前辛房北桥、永定河特大桥向南行驶49.4千米向右前方行驶进入匝道",\n            "road": "G3京台高速",\n            "distance": "49370",\n            "orientation": "南",\n            "duration": "1805"\n          },\n          {\n            "instruction": "沿G3京台高速出口途径G3京台高速、黄泥河大桥、东淀特大桥、吴石槽中桥、西柳河中桥、小白河特大桥、孟家务中桥、文安洼特大桥、大里北中桥、东牛村中桥、子牙河特大桥、关家务中桥、百家洼大桥、烟村中桥、黑龙港特大桥、子牙新河大桥、G2京沪高速向南行驶111.2千米向右前方行驶进入匝道",\n            "road": "G3京台高速出口",\n            "distance": "111168",\n            "orientation": "南",\n            "duration": "3842"\n          },\n          {\n            "instruction": "沿G2京沪高速出口途径G3京台高速、朔黄铁路桥向西南行驶8.7千米靠左沿主路行驶",\n            "road": "G2京沪高速出口",\n            "distance": "8707",\n            "orientation": "西南",\n            "duration": "327"\n          },\n          {\n            "instruction": "沿G3京台高速向南行驶10.1千米靠左沿主路行驶",\n            "road": "G3京台高速",\n            "distance": "10061",\n            "orientation": "南",\n            "duration": "378"\n          },\n          {\n            "instruction": "沿G3京台高速途径津浦铁路桥向西南行驶50.9千米靠左沿主路行驶",\n            "road": "G3京台高速",\n            "distance": "50893",\n            "orientation": "西南",\n            "duration": "1837"\n          },\n          {\n            "instruction": "沿G3京台高速途径减河特大桥、平原互通I桥、马店中桥、徒骇河大桥、赵牛河大桥向南行驶128.2千米靠左",\n            "road": "G3京台高速",\n            "distance": "128161",\n            "orientation": "南",\n            "duration": "4569"\n          },\n          {\n            "instruction": "沿G3京台高速途径韩庄大桥、京台高速济南黄河大桥、王府河大桥、崮山特大桥、夹河大桥、河北特大桥、土门特大桥、大汶河特大桥、贤村水库特大桥、险河大桥、泗河大桥、沂河大桥、大沙河大桥、界河大桥、北沙河大桥、郭河大桥、官木铁路立交桥、新薛河大桥、十字河大桥向南行驶234.1千米靠左",\n            "road": "G3京台高速",\n            "distance": "234066",\n            "orientation": "南",\n            "duration": "8845"\n          },\n          {\n            "instruction": "沿G3京台高速途径枣庄一号高架桥、京杭运河特大桥向东南行驶37.4千米靠左",\n            "road": "G3京台高速",\n            "distance": "37419",\n            "orientation": "东南",\n            "duration": "1405"\n          },\n          {\n            "instruction": "沿G3京台高速途径子房铁路大桥向西南行驶23.6千米靠左沿主路行驶",\n            "road": "G3京台高速",\n            "distance": "23593",\n            "orientation": "西南",\n            "duration": "892"\n          },\n          {\n            "instruction": "沿G2513淮徐高速途径京杭运河大桥、大黄山铁路大桥、陇海铁路大桥、张集大桥、贺楼中沟大桥、洪杨河大桥、废黄河Ⅰ号桥、废黄河II号桥向东南行驶50.9千米向右前方行驶进入匝道",\n            "road": "G2513淮徐高速",\n            "distance": "50892",\n            "orientation": "东南",\n            "duration": "1853"\n          },\n          {\n            "instruction": "沿睢宁西枢纽途径S65徐明高速、S07徐明高速、新濉河大桥向南行驶55.3千米靠左沿主路行驶",\n            "road": "睢宁西枢纽",\n            "distance": "55320",\n            "orientation": "南",\n            "duration": "1983"\n          },\n          {\n            "instruction": "沿徐明高速途径S07徐明高速、新汴河大桥、怀洪新河特大桥向南行驶59.7千米靠左沿主路行驶",\n            "road": "徐明高速",\n            "distance": "59711",\n            "orientation": "南",\n            "duration": "2105"\n          },\n          {\n            "instruction": "沿S07徐明高速途径淮河特大桥向南行驶28.0千米向右前方行驶进入匝道",\n            "road": "S07徐明高速",\n            "distance": "28025",\n            "orientation": "南",\n            "duration": "1025"\n          },\n          {\n            "instruction": "沿明光枢纽向西南行驶680米向左前方行驶",\n            "road": "明光枢纽",\n            "distance": "680",\n            "orientation": "西南",\n            "duration": "39"\n          },\n          {\n            "instruction": "沿明光枢纽途径G36宁洛高速向东南行驶5.0千米靠左",\n            "road": "明光枢纽",\n            "distance": "4987",\n            "orientation": "东南",\n            "duration": "180"\n          },\n          {\n            "instruction": "沿G36宁洛高速途径池河特大桥、林东水库大桥、沙河大桥、新来河大桥向东南行驶86.2千米减速行驶到达收费站",\n            "road": "G36宁洛高速",\n            "distance": "86195",\n            "orientation": "东南",\n            "duration": "2905"\n          },\n          {\n            "instruction": "沿G36宁洛高速途径滁河特大桥、长城圩大桥、官塘河中桥、马汊河大桥向东行驶16.9千米靠左沿主路行驶",\n            "road": "G36宁洛高速",\n            "distance": "16928",\n            "orientation": "东",\n            "duration": "582"\n          },\n          {\n            "instruction": "沿马汊河大桥途径G36宁洛高速、北汊大桥、小江河大桥向东南行驶9.0千米向右前方行驶进入匝道",\n            "road": "马汊河大桥",\n            "distance": "8969",\n            "orientation": "东南",\n            "duration": "378"\n          },\n          {\n            "instruction": "沿G36宁洛高速出口途径浦仪公路、八卦洲大桥向西南行驶3.1千米向右前方行驶进入匝道",\n            "road": "G36宁洛高速出口",\n            "distance": "3106",\n            "orientation": "西南",\n            "duration": "156"\n          },\n          {\n            "instruction": "沿浦仪公路出口向南行驶1.8千米直行进入隧道",\n            "road": "浦仪公路出口",\n            "distance": "1799",\n            "orientation": "南",\n            "duration": "167"\n          },\n          {\n            "instruction": "沿燕子矶长江隧道途径和燕路向东南行驶4.4千米靠左",\n            "road": "燕子矶长江隧道",\n            "distance": "4368",\n            "orientation": "东南",\n            "duration": "419"\n          },\n          {\n            "instruction": "沿和燕快速路向南行驶268米直行进入隧道",\n            "road": "和燕快速路",\n            "distance": "268",\n            "orientation": "南",\n            "duration": "22"\n          },\n          {\n            "instruction": "沿吉祥庵隧道途径和燕快速路向西南行驶2.1千米直行进入隧道",\n            "road": "吉祥庵隧道",\n            "distance": "2139",\n            "orientation": "西南",\n            "duration": "170"\n          },\n          {\n            "instruction": "沿和燕路隧道向西南行驶271米靠左沿当前道路行驶",\n            "road": "和燕路隧道",\n            "distance": "271",\n            "orientation": "西南",\n            "duration": "14"\n          },\n          {\n            "instruction": "沿和燕路隧道途径和燕快速路、红山路向南行驶1.4千米靠左",\n            "road": "和燕路隧道",\n            "distance": "1361",\n            "orientation": "南",\n            "duration": "84"\n          },\n          {\n            "instruction": "沿红山路向南行驶237米直行进入隧道",\n            "road": "红山路",\n            "distance": "237",\n            "orientation": "南",\n            "duration": "28"\n          },\n          {\n            "instruction": "沿红山路隧道途径红山路向南行驶1.5千米靠左沿主路行驶",\n            "road": "红山路隧道",\n            "distance": "1450",\n            "orientation": "南",\n            "duration": "503"\n          },\n          {\n            "instruction": "沿红山路向南行驶173米靠左",\n            "road": "红山路",\n            "distance": "173",\n            "orientation": "南",\n            "duration": "43"\n          },\n          {\n            "instruction": "沿新庄立交途径龙蟠路辅路向南行驶670米向左前方行驶进入主路",\n            "road": "新庄立交",\n            "distance": "670",\n            "orientation": "南",\n            "duration": "80"\n          },\n          {\n            "instruction": "沿龙蟠路向东南行驶857米向右前方行驶进入右侧道路",\n            "road": "龙蟠路",\n            "distance": "857",\n            "orientation": "东南",\n            "duration": "168"\n          },\n          {\n            "instruction": "沿龙蟠路向南行驶966米右转",\n            "road": "龙蟠路",\n            "distance": "966",\n            "orientation": "南",\n            "duration": "155"\n          },\n          {\n            "instruction": "沿龙蟠路途径北京东路向西行驶1.5千米右转",\n            "road": "龙蟠路",\n            "distance": "1484",\n            "orientation": "西",\n            "duration": "335"\n          },\n          {\n            "instruction": "向北行驶34米左转",\n            "distance": "34",\n            "orientation": "北",\n            "duration": "22"\n          },\n          {\n            "instruction": "向北行驶233米左转",\n            "distance": "233",\n            "orientation": "北",\n            "duration": "48"\n          },\n          {\n            "instruction": "向西南行驶49米右转",\n            "distance": "49",\n            "orientation": "西南",\n            "duration": "15"\n          },\n          {\n            "instruction": "向西行驶18米向左前方行驶",\n            "distance": "18",\n            "orientation": "西",\n            "duration": "1"\n          },\n          {\n            "instruction": "向南行驶69米向右前方行驶",\n            "distance": "69",\n            "orientation": "南",\n            "duration": "27"\n          },\n          {\n            "instruction": "向南行驶6米到达目的地",\n            "distance": "6",\n            "orientation": "南",\n            "duration": "1"\n          }\n        ]\n      }\n    ]\n  }\n}', name='maps_direction_driving', id='fd05eeef-9996-45a1-8fac-75c834b6d731', tool_call_id='call_0_aa42068c-0d52-4e84-90a7-d0078d4d495a'), ToolMessage(content='{\n  "route": {\n    "origin": "118.796624,32.059344",\n    "destination": "120.210792,30.246026",\n    "paths": [\n      {\n        "distance": "279948",\n        "duration": "11594",\n        "steps": [\n          {\n            "instruction": "向南行驶85米向右前方行驶",\n            "distance": "85",\n            "orientation": "南",\n            "duration": "37"\n          },\n          {\n            "instruction": "沿和平公园路向南行驶145米左转",\n            "road": "和平公园路",\n            "distance": "145",\n            "orientation": "南",\n            "duration": "59"\n          },\n          {\n            "instruction": "沿北京东路途径兰家庄向东行驶371米左转",\n            "road": "北京东路",\n            "distance": "371",\n            "orientation": "东",\n            "duration": "67"\n          },\n          {\n            "instruction": "沿兰家庄向北行驶136米右转",\n            "road": "兰家庄",\n            "distance": "136",\n            "orientation": "北",\n            "duration": "23"\n          },\n          {\n            "instruction": "沿北京东路向东行驶759米右转",\n            "road": "北京东路",\n            "distance": "759",\n            "orientation": "东",\n            "duration": "115"\n          },\n          {\n            "instruction": "沿龙蟠中路途径内环东线向南行驶408米靠左",\n            "road": "龙蟠中路",\n            "distance": "408",\n            "orientation": "南",\n            "duration": "32"\n          },\n          {\n            "instruction": "沿内环东线向南行驶238米直行进入隧道",\n            "road": "内环东线",\n            "distance": "238",\n            "orientation": "南",\n            "duration": "8"\n          },\n          {\n            "instruction": "沿西安门隧道途径内环东线向南行驶1.8千米靠左",\n            "road": "西安门隧道",\n            "distance": "1770",\n            "orientation": "南",\n            "duration": "105"\n          },\n          {\n            "instruction": "沿内环东线向南行驶284米直行进入隧道",\n            "road": "内环东线",\n            "distance": "284",\n            "orientation": "南",\n            "duration": "11"\n          },\n          {\n            "instruction": "沿通济门隧道途径内环东线、通济门桥向南行驶1.3千米靠左沿主路行驶",\n            "road": "通济门隧道",\n            "distance": "1335",\n            "orientation": "南",\n            "duration": "84"\n          },\n          {\n            "instruction": "沿内环东线向南行驶1.4千米向右前方行驶进入匝道",\n            "road": "内环东线",\n            "distance": "1375",\n            "orientation": "南",\n            "duration": "97"\n          },\n          {\n            "instruction": "沿双桥门立交向南行驶274米靠左",\n            "road": "双桥门立交",\n            "distance": "274",\n            "orientation": "南",\n            "duration": "15"\n          },\n          {\n            "instruction": "沿双桥门立交途径内环南线向东行驶2.3千米直行进入隧道",\n            "road": "双桥门立交",\n            "distance": "2331",\n            "orientation": "东",\n            "duration": "113"\n          },\n          {\n            "instruction": "沿大校场隧道向东行驶386米靠左沿当前道路行驶",\n            "road": "大校场隧道",\n            "distance": "386",\n            "orientation": "东",\n            "duration": "20"\n          },\n          {\n            "instruction": "沿大校场隧道途径内环南线向东行驶1.1千米直行进入中间岔道",\n            "road": "大校场隧道",\n            "distance": "1121",\n            "orientation": "东",\n            "duration": "58"\n          },\n          {\n            "instruction": "沿内环南线向东行驶682米靠左",\n            "road": "内环南线",\n            "distance": "682",\n            "orientation": "东",\n            "duration": "34"\n          },\n          {\n            "instruction": "沿高桥门枢纽途径S87南京支线向东南行驶5.2千米减速行驶到达收费站",\n            "road": "高桥门枢纽",\n            "distance": "5154",\n            "orientation": "东南",\n            "duration": "252"\n          },\n          {\n            "instruction": "沿S87南京支线途径G25长深高速、北河中河大桥、南河特大桥、茶亭河大桥、鲸塘大桥、蠡河大桥向东南行驶141.6千米直行进入隧道",\n            "road": "S87南京支线",\n            "distance": "141564",\n            "orientation": "东南",\n            "duration": "5398"\n          },\n          {\n            "instruction": "沿梯子山隧道途径G25长深高速、合溪新港大桥、长兴新港大桥、王浜头大桥、刘家渡分离立交桥、刘家渡桥、垒珠兜桥、新展桥、龙溪港大桥、龙头山桥、大港桥、二环南路分离立交桥、渔船庄桥、下沈河桥、杨家斗大桥、东苕溪大桥、分庄漾桥向南行驶99.9千米减速行驶到达收费站",\n            "road": "梯子山隧道",\n            "distance": "99891",\n            "orientation": "南",\n            "duration": "3739"\n          },\n          {\n            "instruction": "沿G25长深高速出口向南行驶184米靠左",\n            "road": "G25长深高速出口",\n            "distance": "184",\n            "orientation": "南",\n            "duration": "32"\n          },\n          {\n            "instruction": "沿上塘高架路途径马角洋桥向东南行驶10.1千米向右前方行驶进入匝道",\n            "road": "上塘高架路",\n            "distance": "10082",\n            "orientation": "东南",\n            "duration": "521"\n          },\n          {\n            "instruction": "沿上德立交向东南行驶489米靠左",\n            "road": "上德立交",\n            "distance": "489",\n            "orientation": "东南",\n            "duration": "27"\n          },\n          {\n            "instruction": "沿上德立交向东南行驶131米靠右",\n            "road": "上德立交",\n            "distance": "131",\n            "orientation": "东南",\n            "duration": "14"\n          },\n          {\n            "instruction": "沿上德立交向东北行驶519米靠右",\n            "road": "上德立交",\n            "distance": "519",\n            "orientation": "东北",\n            "duration": "31"\n          },\n          {\n            "instruction": "沿上德立交途径德胜快速路向东行驶2.5千米向右前方行驶进入匝道",\n            "road": "上德立交",\n            "distance": "2533",\n            "orientation": "东",\n            "duration": "144"\n          },\n          {\n            "instruction": "沿石德立交向东行驶222米靠右",\n            "road": "石德立交",\n            "distance": "222",\n            "orientation": "东",\n            "duration": "12"\n          },\n          {\n            "instruction": "沿石德立交途径秋石高架路向南行驶5.5千米向右前方行驶进入匝道",\n            "road": "石德立交",\n            "distance": "5466",\n            "orientation": "南",\n            "duration": "273"\n          },\n          {\n            "instruction": "沿秋石高架路出口途径新业路向东南行驶1.7千米右转",\n            "road": "秋石高架路出口",\n            "distance": "1720",\n            "orientation": "东南",\n            "duration": "215"\n          },\n          {\n            "instruction": "沿富春路向西南行驶60米向右前方行驶",\n            "road": "富春路",\n            "distance": "60",\n            "orientation": "西南",\n            "duration": "12"\n          },\n          {\n            "instruction": "向西行驶52米左转",\n            "distance": "52",\n            "orientation": "西",\n            "duration": "16"\n          },\n          {\n            "instruction": "沿礼宾路向西南行驶105米右转",\n            "road": "礼宾路",\n            "distance": "105",\n            "orientation": "西南",\n            "duration": "9"\n          },\n          {\n            "instruction": "向西北行驶44米左转",\n            "distance": "44",\n            "orientation": "西北",\n            "duration": "16"\n          },\n          {\n            "instruction": "沿中环路向西南行驶32米到达目的地",\n            "road": "中环路",\n            "distance": "32",\n            "orientation": "西南",\n            "duration": "5"\n          }\n        ]\n      }\n    ]\n  }\n}', name='maps_direction_driving', id='cf607cb0-2d85-4533-bf8e-574f4ea0b0b8', tool_call_id='call_1_b79b2e17-2f57-4b3c-89db-629ba972e825'), ToolMessage(content='{\n  "route": {\n    "origin": "120.210792,30.246026",\n    "destination": "121.473667,31.230525",\n    "paths": [\n      {\n        "distance": "174711",\n        "duration": "7573",\n        "steps": [\n          {\n            "instruction": "沿中环路向西南行驶31米左转",\n            "road": "中环路",\n            "distance": "31",\n            "orientation": "西南",\n            "duration": "12"\n          },\n          {\n            "instruction": "向东南行驶35米右转",\n            "distance": "35",\n            "orientation": "东南",\n            "duration": "6"\n          },\n          {\n            "instruction": "沿礼宾路向西南行驶107米向左前方行驶",\n            "road": "礼宾路",\n            "distance": "107",\n            "orientation": "西南",\n            "duration": "5"\n          },\n          {\n            "instruction": "沿富春路向南行驶118米左转",\n            "road": "富春路",\n            "distance": "118",\n            "orientation": "南",\n            "duration": "45"\n          },\n          {\n            "instruction": "沿解放东路向东南行驶595米左转",\n            "road": "解放东路",\n            "distance": "595",\n            "orientation": "东南",\n            "duration": "101"\n          },\n          {\n            "instruction": "沿之江路向东北行驶1.8千米靠右",\n            "road": "之江路",\n            "distance": "1756",\n            "orientation": "东北",\n            "duration": "144"\n          },\n          {\n            "instruction": "沿三新路向北行驶718米向右前方行驶",\n            "road": "三新路",\n            "distance": "718",\n            "orientation": "北",\n            "duration": "88"\n          },\n          {\n            "instruction": "向北行驶382米右转",\n            "distance": "382",\n            "orientation": "北",\n            "duration": "61"\n          },\n          {\n            "instruction": "沿钱江路途径三堡大桥向东北行驶424米靠左",\n            "road": "钱江路",\n            "distance": "424",\n            "orientation": "东北",\n            "duration": "26"\n          },\n          {\n            "instruction": "沿钱江路向东北行驶197米直行进入隧道",\n            "road": "钱江路",\n            "distance": "197",\n            "orientation": "东北",\n            "duration": "8"\n          },\n          {\n            "instruction": "沿钱畔隧道途径钱江路向北行驶1.7千米靠右沿主路行驶",\n            "road": "钱畔隧道",\n            "distance": "1737",\n            "orientation": "北",\n            "duration": "103"\n          },\n          {\n            "instruction": "沿钱江路途径空港高架路入口向北行驶352米靠右",\n            "road": "钱江路",\n            "distance": "352",\n            "orientation": "北",\n            "duration": "60"\n          },\n          {\n            "instruction": "沿空港高架路入口途径钱塘快速路出口、空港高架路、S2杭甬高速向东北行驶6.1千米减速行驶到达收费站",\n            "road": "空港高架路入口",\n            "distance": "6063",\n            "orientation": "东北",\n            "duration": "323"\n          },\n          {\n            "instruction": "沿S2杭甬高速途径沪杭高架二号桥、运输河桥、西沙滩桥、沪杭高架一号桥、上塘河桥、G60沪昆高速、长安公铁桥、长辛塘桥、长山河桥、车河头桥、庄家头桥、长水塘桥、王店公铁桥、沙河港桥、平湖塘桥、中心河桥、白水靖桥、七仙泾桥、向荡港桥、弯良泾桥、南界泾桥、大蒸港桥、斜塘桥、油墩港桥、沈泾塘桥、通波塘桥、洞泾港桥向东北行驶132.5千米靠左沿主路行驶",\n            "road": "S2杭甬高速",\n            "distance": "132496",\n            "orientation": "东北",\n            "duration": "4863"\n          },\n          {\n            "instruction": "沿G60沪昆高速途径蟠龙港桥、春申塘桥向东北行驶2.9千米减速行驶到达收费站",\n            "road": "G60沪昆高速",\n            "distance": "2949",\n            "orientation": "东北",\n            "duration": "143"\n          },\n          {\n            "instruction": "沿G60沪昆高速途径砖新河桥向东北行驶750米靠左沿主路行驶",\n            "road": "G60沪昆高速",\n            "distance": "750",\n            "orientation": "东北",\n            "duration": "59"\n          },\n          {\n            "instruction": "沿G60沪昆高速途径高架桥、七莘立交桥向东北行驶7.0千米靠左沿主路行驶",\n            "road": "G60沪昆高速",\n            "distance": "7013",\n            "orientation": "东北",\n            "duration": "303"\n          },\n          {\n            "instruction": "沿G60沪昆高速向东行驶1.9千米靠左",\n            "road": "G60沪昆高速",\n            "distance": "1904",\n            "orientation": "东",\n            "duration": "92"\n          },\n          {\n            "instruction": "沿沪闵立交途径沪闵高架路向东北行驶2.4千米靠左进入左岔路",\n            "road": "沪闵立交",\n            "distance": "2428",\n            "orientation": "东北",\n            "duration": "119"\n          },\n          {\n            "instruction": "沿沪闵高架路向东北行驶1.4千米靠左进入左岔路",\n            "road": "沪闵高架路",\n            "distance": "1378",\n            "orientation": "东北",\n            "duration": "68"\n          },\n          {\n            "instruction": "沿沪闵高架路向东北行驶3.6千米向右前方行驶进入匝道",\n            "road": "沪闵高架路",\n            "distance": "3633",\n            "orientation": "东北",\n            "duration": "189"\n          },\n          {\n            "instruction": "沿内环漕溪北路立交桥向东北行驶80米靠右",\n            "road": "内环漕溪北路立交桥",\n            "distance": "80",\n            "orientation": "东北",\n            "duration": "9"\n          },\n          {\n            "instruction": "沿内环漕溪北路立交桥途径内环高架路向东行驶363米靠左沿主路行驶",\n            "road": "内环漕溪北路立交桥",\n            "distance": "363",\n            "orientation": "东",\n            "duration": "16"\n          },\n          {\n            "instruction": "沿内环高架路向东北行驶3.1千米靠左沿主路行驶",\n            "road": "内环高架路",\n            "distance": "3079",\n            "orientation": "东北",\n            "duration": "179"\n          },\n          {\n            "instruction": "沿内环高架路向东北行驶829米向右前方行驶进入匝道",\n            "road": "内环高架路",\n            "distance": "829",\n            "orientation": "东北",\n            "duration": "55"\n          },\n          {\n            "instruction": "沿鲁班路立交向东行驶296米靠左",\n            "road": "鲁班路立交",\n            "distance": "296",\n            "orientation": "东",\n            "duration": "15"\n          },\n          {\n            "instruction": "沿鲁班路立交途径南北高架路向北行驶3.0千米向右前方行驶进入匝道",\n            "road": "鲁班路立交",\n            "distance": "2973",\n            "orientation": "北",\n            "duration": "169"\n          },\n          {\n            "instruction": "沿延安东路立交桥向西北行驶244米靠右",\n            "road": "延安东路立交桥",\n            "distance": "244",\n            "orientation": "西北",\n            "duration": "19"\n          },\n          {\n            "instruction": "沿延安东路立交桥向东北行驶495米向右前方行驶进入匝道",\n            "road": "延安东路立交桥",\n            "distance": "495",\n            "orientation": "东北",\n            "duration": "34"\n          },\n          {\n            "instruction": "沿延安东路立交桥向东北行驶331米靠左",\n            "road": "延安东路立交桥",\n            "distance": "331",\n            "orientation": "东北",\n            "duration": "17"\n          },\n          {\n            "instruction": "沿延安东路向东北行驶185米左转",\n            "road": "延安东路",\n            "distance": "185",\n            "orientation": "东北",\n            "duration": "59"\n          },\n          {\n            "instruction": "沿西藏中路向西北行驶325米左转进入主路",\n            "road": "西藏中路",\n            "distance": "325",\n            "orientation": "西北",\n            "duration": "80"\n          },\n          {\n            "instruction": "沿人民大道向西南行驶167米右转",\n            "road": "人民大道",\n            "distance": "167",\n            "orientation": "西南",\n            "duration": "23"\n          },\n          {\n            "instruction": "向西北行驶117米向左前方行驶",\n            "distance": "117",\n            "orientation": "西北",\n            "duration": "30"\n          },\n          {\n            "instruction": "向西南行驶116米左转",\n            "distance": "116",\n            "orientation": "西南",\n            "duration": "32"\n          },\n          {\n            "instruction": "向东行驶45米到达目的地",\n            "distance": "45",\n            "orientation": "东",\n            "duration": "18"\n          }\n        ]\n      }\n    ]\n  }\n}', name='maps_direction_driving', id='7c3039ec-ab82-4c41-bda6-9a1223dbe697', tool_call_id='call_2_5c9a56d4-343f-4727-95bc-039915a3ad83'), AIMessage(content='以下是从北京到上海的自驾游路线规划，途经南京和杭州，总行程约3天，尽量避开拥堵路段：\n\n### **第一天：北京 → 南京**\n- **距离**：约1005公里\n- **预计时间**：约11小时\n- **主要路线**：\n  1. 从北京市出发，沿G3京台高速向南行驶。\n  2. 途经天津、河北、山东、江苏，最终到达南京。\n- **建议**：\n  - 早晨6点前出发，避开早高峰。\n  - 在山东境内可选择G2京沪高速作为备选路线，避开G3京台高速的拥堵。\n\n### **第二天：南京 → 杭州**\n- **距离**：约280公里\n- **预计时间**：约3小时\n- **主要路线**：\n  1. 从南京出发，沿G25长深高速向南行驶。\n  2. 途经湖州，最终到达杭州。\n- **建议**：\n  - 上午9点后出发，避开南京早高峰。\n  - 在杭州可选择绕城高速避开市区拥堵。\n\n### **第三天：杭州 → 上海**\n- **距离**：约175公里\n- **预计时间**：约2.5小时\n- **主要路线**：\n  1. 从杭州出发，沿S2杭甬高速向东北行驶。\n  2. 途经嘉兴，最终到达上海。\n- **建议**：\n  - 早晨7点前出发，避开杭州早高峰。\n  - 进入上海后，选择外环高速避开市区拥堵。\n\n### **注意事项**\n1. **避开高峰时段**：尽量在早晨6点前或晚上8点后行驶，避开早晚高峰。\n2. **实时导航**：使用导航软件（如高德、百度地图）实时查看路况，及时调整路线。\n3. **休息点**：每2-3小时在服务区休息一次，避免疲劳驾驶。\n4. **天气**：提前查看沿途天气，避免恶劣天气影响行程。\n\n如果需要更详细的路线或实时路况更新，可以随时告诉我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 443, 'prompt_tokens': 10622, 'total_tokens': 11065, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 3456}, 'prompt_cache_hit_tokens': 3456, 'prompt_cache_miss_tokens': 7166}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3d5141a69a_prod0225', 'id': '73745844-9024-4e8f-83ef-724ee0c480a4', 'finish_reason': 'stop', 'logprobs': None}, id='run-a57334f0-6b00-4ec4-aa38-6823b65c366c-0', usage_metadata={'input_tokens': 10622, 'output_tokens': 443, 'total_tokens': 11065, 'input_token_details': {'cache_read': 3456}, 'output_token_details': {}})]}
```
以下是从北京到上海的自驾游路线规划，途经南京和杭州，总行程约3天，尽量避开拥堵路段：

### **第一天：北京 → 南京**
- **距离**：约1005公里
- **预计时间**：约11小时
- **主要路线**：
  1. 从北京市出发，沿G3京台高速向南行驶。
  2. 途经天津、河北、山东、江苏，最终到达南京。
- **建议**：
  - 早晨6点前出发，避开早高峰。
  - 在山东境内可选择G2京沪高速作为备选路线，避开G3京台高速的拥堵。

### **第二天：南京 → 杭州**
- **距离**：约280公里
- **预计时间**：约3小时
- **主要路线**：
  1. 从南京出发，沿G25长深高速向南行驶。
  2. 途经湖州，最终到达杭州。
- **建议**：
  - 上午9点后出发，避开南京早高峰。
  - 在杭州可选择绕城高速避开市区拥堵。

### **第三天：杭州 → 上海**
- **距离**：约175公里
- **预计时间**：约2.5小时
- **主要路线**：
  1. 从杭州出发，沿S2杭甬高速向东北行驶。
  2. 途经嘉兴，最终到达上海。
- **建议**：
  - 早晨7点前出发，避开杭州早高峰。
  - 进入上海后，选择外环高速避开市区拥堵。

### **注意事项**
1. **避开高峰时段**：尽量在早晨6点前或晚上8点后行驶，避开早晚高峰。
2. **实时导航**：使用导航软件（如高德、百度地图）实时查看路况，及时调整路线。
3. **休息点**：每2-3小时在服务区休息一次，避免疲劳驾驶。
4. **天气**：提前查看沿途天气，避免恶劣天气影响行程。

如果需要更详细的路线或实时路况更新，可以随时告诉我！